# 🗺️ 14 · Principal component analysis: genes mirror geography
### From 3,571 genes or 3,000 SNPs to a picture you can read: eigenvectors, projections, scree plots, the Lagrangian derivation, the SVD, a simulated map of genetic variation, leukaemia expression and eigen-digits

**BiologyToML** · Part C — Unsupervised learning · Module 14 of 24

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/ml/tutorial/BiologyToML/notebooks/14_pca_dimensionality_reduction.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SciPy, Matplotlib, scikit-learn (digits data set and for checking only) |
| **Prerequisites** | [Modules 03](03_linear_algebra_for_data.ipynb) (vectors, matrices, eigenvectors), [04](04_calculus_and_optimization.ipynb) (derivatives, Lagrange multipliers) and [12](12_clustering_kmeans_hierarchical.ipynb) (unsupervised learning, the Golub data); [Module 02](02_probability_for_biologists.ipynb) for variance and covariance |
| **Estimated time** | 150 min (reading + running + the ✏️ exercises) |
| **02-620 lectures covered** | `011_PCA_mz_031825` slides 1–59 (PCA) — see [`SLIDE_MAP.md`](../SLIDE_MAP.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> Real data: the Golub et al. (1999) leukaemia expression matrix (72 patients × 3,571 genes, loaded with `load_golub`;
> offline, a clearly labelled simulated stand-in is used) and the scikit-learn handwritten-digit images (Section 9).
> SIMULATED data: the two-gene toy, the ten-gene "programmes" example, the stepping-stone genotype grid of Section 7
> (an *analogue* of Novembre et al. 2008 — not their data and not their numbers) and the two "limits" demos.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Explain** why reducing $K$ features to $M \ll K$ helps visualisation and learning, and what PCA keeps: the directions of largest variance.
2. **Compute** principal components by eigendecomposition of the covariance matrix $S=\tfrac1N X^\top X$ with `np.linalg.eigh` — and remember that it returns eigenvalues in *ascending* order.
3. **Project** data onto the top $M$ components, **reconstruct** them, and **explain** why the data must be mean-centred first.
4. **Choose** the number of components with a scree plot and the cumulative fraction of variance explained.
5. **Derive** PCA as a constrained maximisation with a Lagrange multiplier, **verify** the answer by brute force, and **explain** why the SVD is preferred when $K \gg N$.
6. **Interpret** PCA of genotypes (a simulated isolation-by-distance map), of real leukaemia expression (ALL vs AML, loadings, the effect of scaling) and of images (eigen-digits).
7. **Recognise** PCA's limits — linearity, variance ≠ biological relevance, scaling choices — and treat PCA plots as hypothesis-generating.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, toolkit, the Golub data | 5 min |
| 1 · Why reduce dimension? | the size of biological data; genes are redundant | 10 min |
| 2 · The 2-D → 1-D toy | rotate, project, keep the axis of largest variance | 15 min |
| 3 · Step 1: principal components | covariance matrix, `eigh`, eigenvalues = variances | 15 min |
| 4 · Step 2: projection | coordinates, scores, reconstruction, why centring matters | 15 min |
| 5 · How many components? | scree plot, cumulative variance explained | 10 min |
| 6 · Why it works | Lagrangian, brute-force check, orthogonality, SVD | 20 min |
| 7 · Genes mirror geography | a simulated stepping-stone population → PC map | 20 min |
| 8 · Leukaemia expression | Golub PCA, loadings, raw vs standardised | 15 min |
| 9 · Eigen-digits | images as vectors, reconstruction from 4 numbers | 10 min |
| 10 · Limits of PCA | a cell-cycle loop, a batch effect, honest interpretation | 10 min |
| ✅ / 🧠 / 🚀 | checkpoint, discussion, going further | 5 min |

## 🧫 The biological question

In 2008, John Novembre and colleagues took the genotypes of **1,387 Europeans at 197,146 SNPs** — a table with more
than 270 million entries — and asked a simple question: *is there any structure in this table that you could see with
your own eyes?* They reduced each person from 197,146 numbers to **two** numbers with **principal component analysis
(PCA)**. When every person was drawn as a dot at those two coordinates and coloured by country of origin, the cloud of
dots looked strikingly like **a map of Europe** (02-620 `011_PCA` slides 5 and 45). Nobody told the algorithm where anyone
came from: geography emerged because people who live near each other share more recent ancestors, so their genotypes
are more similar — *genes mirror geography*.

The same tool is everywhere in biology:

* **Genetics.** PCs of a genotype matrix summarise ancestry. Genome-wide association studies ([Module 06](06_linear_regression_gwas.ipynb)) routinely add
  the top PCs as covariates so that ancestry differences between cases and controls are not mistaken for disease
  associations (Price *et al.* 2006).
* **Expression.** A first PCA plot of an RNA-seq or microarray experiment shows at a glance whether samples group by
  biology (tumour type, treatment) — or by technical batch. You will see both in the real Golub leukaemia data.
* **Images.** A microscopy image with thousands of pixels can be summarised by a handful of PC coefficients
  ("eigen-images").

PCA is *unsupervised* ([Module 12](12_clustering_kmeans_hierarchical.ipynb)): it never sees labels. It finds new axes — weighted combinations of the original
features — along which the samples vary the most. This notebook builds it from scratch, proves why it works, and shows
you how to read (and how *not* to over-read) a PCA plot.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this module needs only a CPU.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("sklearn", "scikit-learn")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py).
`load_golub` reads the real leukaemia data (with an offline fallback), `standardize` z-scores columns, and `CLASS_COLORS` gives the course's colour-blind-safe class colours (ALL blue, AML red). Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import os
import urllib.request
import numpy as np
import pandas as pd

REPO_DATA_URL = "https://raw.githubusercontent.com/xulabs/edu/main/ml/tutorial/BiologyToML/datasets/{name}"


CACHE_DIR = os.environ.get("BIO2ML_CACHE", "course_data")


def fetch_course_file(name: str, timeout: float = 60.0) -> str | None:
    """Return a local path to the course data file ``name``, or ``None`` if it cannot be found.

    Looks in the repository's ``datasets/`` folder first (when the notebook runs inside a
    clone of xulabs/edu), then in the download cache, and finally downloads it from GitHub.
    """
    for base in ("../datasets", "datasets", "ml/tutorial/BiologyToML/datasets", CACHE_DIR):
        path = os.path.join(base, name)
        if os.path.exists(path) and os.path.getsize(path) > 0:
            return path
    dest = os.path.join(CACHE_DIR, name)
    os.makedirs(CACHE_DIR, exist_ok=True)
    try:
        with urllib.request.urlopen(REPO_DATA_URL.format(name=name), timeout=timeout) as r, \
                open(dest + ".part", "wb") as f:
            f.write(r.read())
        os.replace(dest + ".part", dest)
        return dest
    except Exception as e:  # noqa: BLE001 — any network problem → offline fallback
        print(f"⚠️  Could not obtain {name} ({type(e).__name__}); using a simulated stand-in instead.")
        return None


def simulate_golub_like(rng: np.random.Generator) -> dict:
    """Simulated stand-in for the Golub data: 72 samples (47 'ALL', 25 'AML') × 3,571 genes.

    Log10-scale expression with gene-specific means, correlated gene modules and
    60 genes that differ between the classes.  Same keys as ``load_golub``.
    """
    n_all, n_aml, J = 47, 25, 3571
    y = np.r_[np.zeros(n_all, int), np.ones(n_aml, int)]
    base = rng.normal(2.6, 0.45, J)
    modules = rng.normal(0, 1, (len(y), 20))                  # 20 hidden "programmes"
    loading = rng.normal(0, 0.15, (20, J)) * (rng.random((20, J)) < 0.1)
    X = base + modules @ loading + rng.normal(0, 0.25, (len(y), J))
    informative = rng.choice(J, 60, replace=False)
    X[:, informative] += np.outer(y - 0.5, rng.choice([-1, 1], 60) * rng.uniform(0.4, 0.9, 60))
    is_train = np.r_[np.arange(n_all) < 27, np.arange(n_aml) < 11]
    return dict(X=X, y=y, labels=np.where(y == 1, "AML", "ALL"), genes=np.array([f"gene{j:04d}" for j in range(J)]),
                is_train=is_train, sample_id=np.arange(1, len(y) + 1), source="SIMULATED stand-in (offline)")


def load_golub() -> dict:
    """Golub et al. (1999) leukaemia expression data, preprocessed as in Dudoit et al. (2002).

    Returns a dict with ``X`` (72 × 3,571, log10 expression), ``y`` (0 = ALL, 1 = AML),
    ``labels``, ``genes`` (Affymetrix probe IDs), ``is_train`` (the original 38/34 split),
    ``sample_id`` and ``source``.  Falls back to ``simulate_golub_like`` when offline.

    Raw intensities were clipped to [100, 16000] before the log10, so the value 2.0 means
    "at or below the detection limit" (about 20 % of all entries).  Some genes are therefore
    constant within a class — per-class variance estimates need a small floor.
    """
    path = fetch_course_file("golub_leukemia.csv.gz")
    if path is None:
        return simulate_golub_like(np.random.default_rng(1999))
    df = pd.read_csv(path)
    return dict(X=df.iloc[:, 3:].to_numpy(float), y=(df["label"] == "AML").to_numpy(int),
                labels=df["label"].to_numpy(), genes=np.array(df.columns[3:]),
                is_train=(df["golub_split"] == "train").to_numpy(), sample_id=df["sample_id"].to_numpy(),
                source="Golub et al. 1999 (real data)")


def standardize(X, mean=None, sd=None):
    """Z-score the columns of X. Pass the training mean/sd to transform test data the same way.

    Returns ``(Z, mean, sd)``; constant columns get sd = 1 so they map to 0.
    """
    X = np.asarray(X, float)
    mean = X.mean(axis=0) if mean is None else mean
    sd = X.std(axis=0) if sd is None else sd
    sd = np.where(sd > 0, sd, 1.0)
    return (X - mean) / sd, mean, sd


CLASS_COLORS = ["#1f77b4", "#d62728", "#2ca02c", "#9467bd", "#ff7f0e", "#8c564b"]


print("🧰 Toolkit loaded:", ", ".join(['fetch_course_file', 'simulate_golub_like', 'load_golub', 'standardize']))

---
## 1 · Why reduce dimension?

> 📍 **Where we are.** In [Module 12](12_clustering_kmeans_hierarchical.ipynb) you clustered the Golub patients using
> thousands of genes at once, and in [Module 13](13_mixture_models_em.ipynb) you modelled data as mixtures of Gaussians. Both work in the *original* feature space. PCA asks a different
> unsupervised question: can we describe each sample with **far fewer, new features** that keep most of the information?

The PCA slides call the number of features **$K$** (the rest of the course uses $J$); we follow the slides in this
module. A data set is a matrix $X$ with $N$ rows (samples: patients, individuals, cells) and $K$ columns (features:
genes, SNPs, pixels). The slides give two reasons to reduce $K$ to some $M \ll K$ (slides 2–3):

1. **Visualisation.** We can look at 2 or 3 dimensions, not 3,571.
2. **Learnability.** A model with one weight per feature has more unknowns than samples when $K > N$ — exactly why
   least squares broke down in [Module 06](06_linear_regression_gwas.ipynb) and needed ridge/lasso in
   [Module 07](07_ridge_lasso_sparse_regression.ipynb). Fewer, well-chosen features make models
   easier to fit.

Reduction is possible only if the features are **redundant** — if many of them go up and down together. Let us measure
the size of the problem and the redundancy in real data.

In [ ]:
# ── 1.1 How big are biological data sets? ──
golub = load_golub()
print("Source:", golub["source"])
X_golub, y_golub = golub["X"], golub["y"]
N, K = X_golub.shape
print(f"Golub leukaemia data: N = {N} patients × K = {K:,} genes "
      f"({np.sum(y_golub == 0)} ALL, {np.sum(y_golub == 1)} AML)")
print(f"  scatter plots needed to look at every pair of genes: K(K−1)/2 = {K * (K - 1) // 2:,}")
K_eu = 197_146
print(f"Novembre et al. (2008): N = 1,387 Europeans × K = {K_eu:,} SNPs "
      f"→ {K_eu * (K_eu - 1) // 2:,} pairs of SNPs")

In [ ]:
# ── 1.2 Genes are redundant: correlations between random pairs of genes ──
rng = np.random.default_rng(SEED + 1)
Z_golub = standardize(X_golub)[0]                        # z-scored genes: correlation = mean of products
pairs = rng.integers(0, K, size=(5000, 2))
pairs = pairs[pairs[:, 0] != pairs[:, 1]]
r_real = np.mean(Z_golub[:, pairs[:, 0]] * Z_golub[:, pairs[:, 1]], axis=0)
# Null model: shuffle the patients of the second gene → any shared pattern is destroyed
shuffled = np.stack([rng.permutation(Z_golub[:, b]) for b in pairs[:, 1]], axis=1)
r_null = np.mean(Z_golub[:, pairs[:, 0]] * shuffled, axis=0)
print(f"|r| > 0.5 for {np.mean(np.abs(r_real) > 0.5):.1%} of {len(pairs):,} random gene pairs "
      f"(shuffled null: {np.mean(np.abs(r_null) > 0.5):.2%})")

fig, ax = plt.subplots(figsize=(6.5, 3))
ax.hist(r_null, bins=60, range=(-1, 1), alpha=0.6, color="0.6", label="shuffled patients (no shared signal)")
ax.hist(r_real, bins=60, range=(-1, 1), alpha=0.6, color="tab:blue", label="real pairs of genes")
ax.set_xlabel("Pearson correlation between two genes across 72 patients"); ax.set_ylabel("number of gene pairs")
ax.set_title("Real genes co-vary far more than chance: the data are redundant"); ax.legend()
plt.tight_layout(); plt.show()

🔍 **What just happened?** With 3,571 genes there are more than six million gene pairs — nobody can look at them all.
But the genes are not independent: real gene pairs have a much wider spread of correlations than the shuffled null,
and a few per cent of random pairs are strongly correlated ($|r|>0.5$, see the printout) — extrapolated to all
6.4 million pairs, that is on the order of 200,000 strongly correlated gene pairs — whereas essentially no shuffled
pair is.
Genes in the same pathway, or genes that respond to the same cell type or the same technical factor, rise and fall
together. Redundancy means that a few well-chosen *combinations* of genes can carry most of the information — which
is exactly what PCA looks for.

✏️ **Your turn.** Replace the threshold 0.5 by 0.8. What fraction of real pairs remains? What does a correlation of 0.8
between two genes suggest about whether you need both of them to describe the patients?

### 📑 Slide connection — `011_PCA_mz_031825`, slides 2–6

- **What the slides show.** High-dimensional data ($N$ samples × $K$ features, large $K$) are hard to visualise and hard to fit models to (slide 2). Dimensionality reduction is a type of unsupervised learning that turns the $N\times K$ matrix into an $N\times M$ matrix with $M\ll K$ transformed features that capture the most important dimensions (slide 3). Genotype example: 200K SNPs → two linearly transformed features that capture the largest variation (slide 4); Novembre et al. 2008: 197,146 SNPs, 1,387 Europeans, each dot an individual on the top two PCs (slide 5). Slide 6 announces the plan: the concept, how to execute PCA, and why it works.
- **What this notebook adds.** The actual size of a real expression data set and a quantitative look at *redundancy* (correlations of random gene pairs against a shuffled null) — the property that makes reduction possible.
- **Exam tip.** Be able to give both motivations (visualisation and learnability, with the $K>N$ problem of Module 06) and to say why PCA is *unsupervised*: it uses only $X$, never the labels.

---
## 2 · The 2-D → 1-D toy: rotate, project, keep the widest axis

> 📍 **Where we are.** Before 3,571 genes, two. The slides (7–13) explain PCA with a cloud of points in two dimensions
> that we want to describe with *one* number per point.

Imagine two genes, A and B, that belong to the same signalling pathway. In each cell both respond to the pathway's
activity (with different strengths) plus independent noise, so the cloud of cells is an elongated ellipse. To describe
each cell by one number, we choose a direction — a line through the centre of the cloud — and record where each cell's
**shadow** falls on that line (its *projection*). The slides rotate the axes to new axes $y_1, y_2$ and ask: *which of
the two new 1-D data sets is better?* Their answer: the one with **higher variability** (slide 12) — a long axis keeps
cells apart, a short axis squashes them together.

🧮 **Math refresher — projecting onto a direction.** A direction in the plane is a **unit vector**
$u = (\cos\theta, \sin\theta)$: an arrow of length $\lVert u\rVert = \sqrt{\cos^2\theta + \sin^2\theta} = 1$ at angle
$\theta$. For a (centred) data point $x_i = (x_{i1}, x_{i2})$ the **dot product**

$$z_i = x_i^\top u = x_{i1}\cos\theta + x_{i2}\sin\theta$$

*In words:* the position of the point's shadow along the line — "how many steps of length 1 in direction $u$" (Module 03).
The variance of the shadows is $\frac{1}{N}\sum_{i=1}^{N} z_i^2$ (their mean is 0 because the data are centred), where
$\sum_{i=1}^{N}$ means "add up over all $N$ samples".

In [ ]:
# ── 2.1 Two co-regulated genes: a 2-D toy data set (SIMULATED) ──
rng = np.random.default_rng(SEED + 2)
n_cells = 300
activity = rng.normal(0, 1, n_cells)                          # hidden pathway activity of each cell
gene_a = 1.0 * activity + rng.normal(0, 0.35, n_cells)        # both genes respond to the pathway …
gene_b = 0.7 * activity + rng.normal(0, 0.35, n_cells)        # … with different strengths, plus noise
X2 = np.column_stack([gene_a, gene_b]) + np.array([3.0, 9.0])  # log-expression around 3 and 9
X2c = X2 - X2.mean(axis=0)                                     # mean-centre (Section 4 explains why)
print("X2 shape:", X2.shape, "| column means before centring:", X2.mean(axis=0).round(2),
      "| after:", np.abs(X2c.mean(axis=0)).max().round(12))
print("correlation between gene A and gene B:", np.corrcoef(X2[:, 0], X2[:, 1])[0, 1].round(3))

In [ ]:
# ── 2.2 Sweep the projection angle and record the variance of the shadows ──
def projected_variance(Xc, theta):
    """Variance of the projections of centred 2-D data onto the unit vector at angle theta (radians)."""
    u = np.array([np.cos(theta), np.sin(theta)])
    z = Xc @ u                                  # one number per cell: x_iᵀu
    return np.mean(z ** 2)                      # data are centred, so the mean of z is 0


thetas = np.radians(np.arange(0, 180, 1.0))     # directions 0°, 1°, …, 179° (180° is the same line as 0°)
pv = np.array([projected_variance(X2c, t) for t in thetas])
best, worst = thetas[np.argmax(pv)], thetas[np.argmin(pv)]
print(f"widest direction  y1: {np.degrees(best):.0f}°  → variance {pv.max():.3f}")
print(f"narrowest direction y2: {np.degrees(worst):.0f}°  → variance {pv.min():.3f}  "
      f"(difference {np.degrees(worst - best):+.0f}°)")

fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
axes[0].scatter(X2c[:, 0], X2c[:, 1], s=8, alpha=0.5, color="0.4")
for t, col, lab in [(best, "tab:red", "y1 (widest)"), (worst, "tab:blue", "y2 (narrowest)")]:
    axes[0].plot([-3.5 * np.cos(t), 3.5 * np.cos(t)], [-3.5 * np.sin(t), 3.5 * np.sin(t)], color=col, lw=2, label=lab)
axes[0].set_aspect("equal"); axes[0].legend(loc="upper left")
axes[0].set_xlabel("gene A (centred log-expression)"); axes[0].set_ylabel("gene B (centred)")
axes[0].set_title("SIMULATED co-regulated genes")
axes[1].plot(np.degrees(thetas), pv, color="k")
axes[1].axvline(np.degrees(best), color="tab:red", ls="--"); axes[1].axvline(np.degrees(worst), color="tab:blue", ls="--")
axes[1].set_xlabel("direction θ of the projection line (degrees)"); axes[1].set_ylabel("variance of projections")
axes[1].set_title("one direction keeps the most variance")
for k, (t, col) in enumerate([(best, "tab:red"), (worst, "tab:blue")]):
    z = X2c @ np.array([np.cos(t), np.sin(t)])
    axes[2].scatter(z, np.full_like(z, k) + rng.uniform(-0.15, 0.15, len(z)), s=6, alpha=0.5, color=col)
axes[2].set_yticks([0, 1], ["on y1", "on y2"]); axes[2].set_xlabel("position of the shadow (1-D data)")
axes[2].set_title("the two 1-D data sets of slide 11")
plt.tight_layout(); plt.show()

🔍 **What just happened?** We tried all 180 possible lines through the centre of the cloud. The variance of the
projected cells is a smooth curve with **one maximum** (the long axis of the ellipse, $y_1$) and **one minimum** exactly
**90° away** ($y_2$), as the printout shows. On $y_1$ the cells stay spread out — their order along the line still tells
you which cells have high and which have low pathway activity. On $y_2$ they are squashed together: what is left is
mostly noise. Keeping $y_1$ is "projecting to the axis that results in higher variability" (slide 12).

But a brute-force sweep is only possible in 2-D. In $K$ dimensions there are infinitely many directions in a
$K$-dimensional sphere of possibilities. The next section finds the best direction in one step — with an
**eigendecomposition**.

✏️ **Your turn.** Change the noise of gene B from 0.35 to 1.0 (so the genes are less correlated) and re-run 2.1–2.2.
How do the maximum and the minimum of the curve change? What would the curve look like if the two genes were
completely uncorrelated and had equal variance?

### 📑 Slide connection — `011_PCA_mz_031825`, slides 7–13

- **What the slides show.** Reduce 2-D data $(x_1,x_2)$ to 1-D (slides 7–8); define new features $y_1, y_2$ (rotated axes, slide 9); rotate the data and project onto one of the new axes (slide 10); compare the two 1-D data sets (slide 11) and keep the axis with higher variability, $y_1$ (slide 12).
- **What this notebook adds.** A complete sweep over all directions, showing that projected variance is a smooth function of the angle with its maximum and minimum 90° apart — the two principal axes we compute exactly in Section 3.
- **Exam tip.** 'Better' means **larger variance of the projected data**. The best and worst directions are perpendicular — a first hint that principal components are orthogonal.

---
## 3 · Step 1 — principal components by eigendecomposition

> 📍 **Where we are.** Section 2 found the widest direction by trying every angle. The slides' recipe (slide 17) has
> two steps: **Step 1** find the principal components with an *eigendecomposition*; **Step 2** project the data onto
> the ones you choose (Section 4).

🧮 **Math refresher — the covariance matrix.** Write the centred data as an $N\times K$ matrix $X$ whose $i$-th row is
$x_i^\top$ (the slides write $X = [X_1^\top; \dots; X_N^\top]$). The **sample covariance matrix** is the $K\times K$ matrix

$$S = \frac{1}{N} X^\top X, \qquad S_{jk} = \frac{1}{N}\sum_{i=1}^{N} x_{ij}\,x_{ik}.$$

*In words:* entry $(j,k)$ is the average product of feature $j$ and feature $k$ over the samples — the **covariance**
of the two features (Module 02); on the diagonal ($j=k$) it is the **variance** of feature $j$. $X^\top$ ("X transpose")
is $X$ with rows and columns swapped, so $X^\top X$ multiplies every column with every other column (Module 03). $S$ is
**symmetric** ($S_{jk} = S_{kj}$).

🧮 **Math refresher — eigenvectors.** A non-zero vector $u$ is an **eigenvector** of $S$ with **eigenvalue** $\lambda$ if

$$S\,u = \lambda\, u .$$

*In words:* multiplying by $S$ does not turn $u$; it only stretches it by the factor $\lambda$. A symmetric $K\times K$
matrix has $K$ eigenvectors that can be chosen of length 1 and mutually perpendicular, so it can be written as
(slides 18–20)

$$S = U D U^\top, \qquad U = [u_1, \dots, u_K],\quad U^\top U = I,\quad D = \mathrm{diag}(\lambda_1,\dots,\lambda_K).$$

*In words:* $U$'s columns are the eigenvectors (an orthogonal matrix: $u_j^\top u_k = 0$ for $j\neq k$ and $u_k^\top u_k = 1$),
and the diagonal matrix $D$ holds the eigenvalues. For a covariance matrix all $\lambda_k \ge 0$.

**The PCA recipe (slides 21–23, 31):** mean-centre the data; compute $S = \frac1N X^\top X$; eigendecompose
$S = UDU^\top$; the eigenvectors with the **largest eigenvalues** are the principal components (PCs).

In [ ]:
# ── 3.1 The sample covariance matrix S = XᵀX / N ──
N2 = len(X2c)
S2 = X2c.T @ X2c / N2                                    # 2 × 2
print("S = XᵀX / N =\n", S2.round(4))
print("matches np.cov(X, bias=True):", np.allclose(S2, np.cov(X2, rowvar=False, bias=True)))
print(f"⚠️ np.cov's default divides by N−1: S[0,0] = {np.cov(X2, rowvar=False)[0, 0]:.4f} instead of {S2[0, 0]:.4f}")

In [ ]:
# ── 3.2 Eigendecomposition with np.linalg.eigh — ascending order! ──
evals, evecs = np.linalg.eigh(S2)
print("eigh returns the eigenvalues in ASCENDING order:", evals.round(4))
order = np.argsort(evals)[::-1]                            # indices from largest to smallest
lam2, U2 = evals[order], evecs[:, order]                   # reorder eigenvalues AND the matching columns
U2 = U2 * np.sign(U2[np.argmax(np.abs(U2), axis=0), range(2)])   # sign convention: largest entry positive
print("sorted, largest first:", lam2.round(4))
for k in range(2):
    print(f"S u{k + 1} = {np.round(S2 @ U2[:, k], 4)}   λ{k + 1} u{k + 1} = {np.round(lam2[k] * U2[:, k], 4)}")
print("UᵀU =\n", (U2.T @ U2).round(10))
print("U D Uᵀ reproduces S:", np.allclose(U2 @ np.diag(lam2) @ U2.T, S2))

In [ ]:
# ── 3.3 λ1 is the variance along PC1 — the same answer as the brute-force sweep ──
z1, z2 = X2c @ U2[:, 0], X2c @ U2[:, 1]                     # projections on PC1 and PC2
angle_pc1 = np.degrees(np.arctan2(U2[1, 0], U2[0, 0])) % 180
print(f"Var(X u1) = {np.mean(z1 ** 2):.4f}   λ1 = {lam2[0]:.4f}")
print(f"Var(X u2) = {np.mean(z2 ** 2):.4f}   λ2 = {lam2[1]:.4f}")
print(f"PC1 points at {angle_pc1:.1f}°;  the sweep's best angle was {np.degrees(best):.0f}° "
      f"(variance {pv.max():.4f})")
print(f"λ1 + λ2 = {lam2.sum():.4f} = total variance trace(S) = {np.trace(S2):.4f}; "
      f"PC1 keeps {lam2[0] / lam2.sum():.1%} of it")

fig, ax = plt.subplots(figsize=(4.6, 3.8))
ax.scatter(X2c[:, 0], X2c[:, 1], s=8, alpha=0.4, color="gold", edgecolor="0.6", linewidth=0.2)
for k, col in enumerate(["tab:red", "tab:blue"]):
    vec = 2 * np.sqrt(lam2[k]) * U2[:, k]                    # arrow of length 2·√λ_k along u_k
    ax.annotate("", xy=vec, xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", color=col, lw=2))
    ax.text(*(vec * 1.12), f"PC{k + 1}\nλ{k + 1}={lam2[k]:.2f}", color=col, fontsize=8, ha="center")
ax.set_aspect("equal"); ax.set_xlabel("gene A (centred)"); ax.set_ylabel("gene B (centred)")
ax.set_title("Eigenvectors = principal axes\n(arrows: 2·√λ, i.e. two standard deviations)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** One call to `eigh` gave both principal axes at once. After reversing its ascending order,
PC1's direction agrees with the brute-force sweep (to within the sweep's 1° grid), and the eigenvalue $\lambda_1$ is
*exactly* the variance of the data projected on $u_1$ — the slides' "variance explained: $\lambda_1$" (slide 27). The two
eigenvalues add up to the total variance (the sum of the diagonal of $S$, called the **trace**): PCA does not create or
destroy variance, it re-distributes it so that the first axis gets as much as possible.

⚠️ **Three classic pitfalls.**
1. `np.linalg.eigh` returns eigenvalues in **ascending** order — the *last* column is PC1. Always sort.
2. An eigenvector is only defined up to its **sign**: if $Su=\lambda u$ then also $S(-u)=\lambda(-u)$. Different
   computers may return $-u_1$; a PCA plot can come out mirrored. We fix a convention (largest entry positive).
3. `np.cov` divides by $N-1$, the slides by $N$. Both give the same eigenvectors; eigenvalues differ by $N/(N-1)$.

✏️ **Your turn.** Compute the eigenvalues of the printed $2\times2$ matrix $S$ *by hand* using the characteristic
equation $\lambda^2 - (S_{11}+S_{22})\lambda + (S_{11}S_{22}-S_{12}^2) = 0$ and check them against the printout.

### 📑 Slide connection — `011_PCA_mz_031825`, slides 17–31

- **What the slides show.** Two steps: find the PCs by eigendecomposition, then project by inner products (slide 17). Review: a $K\times K$ symmetric $S = UDU^\top$ with $U^\top U = I$, computed with `numpy.linalg.eigh` (slides 18–20). Step 1: centre the data, compute $S=\frac1N X^\top X$, eigendecompose, pick the $M$ eigenvectors with the largest eigenvalues (slides 21–23, 31). On the 2-D toy, $u_1, u_2$ are orthonormal unit vectors and $\lambda_k$ is the variance explained by PC $k$ (slides 24–28); pick the dimension with the largest eigenvalue (slides 29–30).
- **What this notebook adds.** Every claim checked numerically: $Su_k=\lambda_k u_k$, $U^\top U=I$, $UDU^\top=S$, $\mathrm{Var}(Xu_1)=\lambda_1$, and agreement with the brute-force sweep of Section 2. Plus the practical traps the slides do not mention: `eigh`'s ascending order, sign ambiguity, and $N$ vs $N-1$.
- **Exam tip.** If asked for 'the first PC' from `eigh` output, take the eigenvector of the **largest** eigenvalue (the *last* column). For a $2\times2$ covariance, be able to find $\lambda$ from the characteristic equation; $\lambda_1+\lambda_2 = \mathrm{trace}(S)$ and $\lambda_1\lambda_2 = \det S$ are quick checks.

---
## 4 · Step 2 — projecting the data onto the principal components

> 📍 **Where we are.** We have the new axes $u_1, \dots, u_K$. Step 2 (slides 32–40) re-describes every sample in the
> new coordinate system and then keeps only the first $M$ coordinates.

**Coordinates are dot products (slides 37–38).** In the usual system, $x = (2, 1)$ means "walk 2 steps along
$u_1 = (1,0)$ and 1 step along $u_2 = (0,1)$". In *any* orthonormal system $v_1, v_2$, the number of steps along $v_k$
is the dot product $x^\top v_k = \lVert x\rVert\,\lVert v_k\rVert\cos\theta = \lVert x\rVert\cos\theta$ (the slide-32
notes), because $v_k$ has length 1. Stacking all samples:

$$X_p = X\,[u_1, \dots, u_M], \qquad (N\times M) = (N\times K)\,(K\times M).$$

*In words:* the **PC scores** of sample $i$ are $x_i^\top u_1, \dots, x_i^\top u_M$ — how far its shadow falls along
each of the first $M$ principal axes (slide 40). Because $U$ is orthogonal we can also walk back:
$x_i = \sum_{k=1}^{K} (x_i^\top u_k)\,u_k$, and keeping only $M$ terms gives the best $M$-dimensional approximation.

In [ ]:
# ── 4.1 Coordinates: x = (2, 1) in two coordinate systems ──
x = np.array([2.0, 1.0])
E = np.eye(2)                                         # the usual axes u1 = (1,0), u2 = (0,1) as columns
print("usual axes   → steps (xᵀu1, xᵀu2) =", x @ E)
phi = np.radians(30)
V_rot = np.array([[np.cos(phi), -np.sin(phi)],        # columns v1, v2: the usual axes turned by 30°
                  [np.sin(phi), np.cos(phi)]])
steps = x @ V_rot                                     # (xᵀv1, xᵀv2)
print("rotated axes → steps (xᵀv1, xᵀv2) =", steps.round(4))
print("walking back: steps[0]·v1 + steps[1]·v2 =", (steps[0] * V_rot[:, 0] + steps[1] * V_rot[:, 1]).round(12))
print("v1ᵀv2 =", round(float(V_rot[:, 0] @ V_rot[:, 1]), 12), "| length of x is unchanged:",
      np.linalg.norm(x).round(4), np.linalg.norm(steps).round(4))

In [ ]:
# ── 4.2 Project the toy data onto PC1 and reconstruct ──
scores2 = X2c @ U2                                    # N × 2: the scores on PC1 and PC2
X2_back = scores2 @ U2.T                              # walk back with all K = 2 components
X2_m1 = np.outer(scores2[:, 0], U2[:, 0])             # keep only M = 1: x̂_i = (x_iᵀu1) u1
print("exact reconstruction with all PCs:", np.allclose(X2_back, X2c))
print(f"M = 1: mean squared error per cell = {np.mean(np.sum((X2c - X2_m1) ** 2, axis=1)):.4f}  (λ2 = {lam2[1]:.4f})")

fig, ax = plt.subplots(figsize=(5, 3.8))
ax.scatter(X2c[:, 0], X2c[:, 1], s=8, alpha=0.3, color="0.5", label="cells (centred)")
show = rng.choice(N2, 25, replace=False)
for i in show:                                        # perpendicular drop of each point onto the PC1 line
    ax.plot([X2c[i, 0], X2_m1[i, 0]], [X2c[i, 1], X2_m1[i, 1]], color="tab:blue", lw=0.8)
ax.scatter(X2_m1[show, 0], X2_m1[show, 1], s=14, color="tab:red", zorder=3, label="projection on PC1")
ax.axline((0, 0), U2[:, 0], color="tab:red", lw=1)
ax.set_aspect("equal"); ax.legend(loc="upper left")
ax.set_xlabel("gene A (centred)"); ax.set_ylabel("gene B (centred)")
ax.set_title("Step 2: each cell is replaced by its shadow on PC1")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The point $(2,1)$ has different *coordinates* in the rotated system but the same length:
rotating axes changes the description, not the data. For the toy cells, the two scores reproduce every cell exactly;
keeping only the PC1 score replaces each cell by its shadow (red dots), and the average squared error of that
replacement equals $\lambda_2$ — the variance along the axis we threw away. That is the general rule: **the
reconstruction error of PCA with $M$ components is the sum of the discarded eigenvalues** (Section 9 checks it on images).

The slides end Step 2 with a question: *"Remember to mean centre the data before projection (why?)"* (slide 36). Let's
find out by forgetting to do it.

In [ ]:
# ── 4.3 ⚠️ What goes wrong if you forget to centre? ──
S_unc = X2.T @ X2 / N2                                # "covariance" of the UNcentred data (means ≈ 3 and 9)
w_unc, V_unc = np.linalg.eigh(S_unc)
u_unc = V_unc[:, -1] * np.sign(V_unc[:, -1].sum())    # top eigenvector (last column!)
mean_dir = X2.mean(axis=0) / np.linalg.norm(X2.mean(axis=0))


def angle_deg(v):
    """Angle of a 2-D vector in degrees, folded into [0°, 180°)."""
    return np.degrees(np.arctan2(v[1], v[0])) % 180


print(f"top eigenvector of the UNcentred XᵀX/N points at {angle_deg(u_unc):5.1f}°")
print(f"direction of the mean vector (3, 9) points at    {angle_deg(mean_dir):5.1f}°")
print(f"true PC1 of the centred data points at           {angle_deg(U2[:, 0]):5.1f}°")
print(f"variance of the centred cells along the uncentred 'PC1': {np.mean((X2c @ u_unc) ** 2):.3f} "
      f"(true λ1 = {lam2[0]:.3f})")

fig, ax = plt.subplots(figsize=(4.6, 3.8))
ax.scatter(X2[:, 0], X2[:, 1], s=6, alpha=0.4, color="0.5")
L_unc = 0.8 * np.linalg.norm(X2.mean(axis=0))            # arrow length: most of the way to the cloud
ax.annotate("", xy=L_unc * u_unc, xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", color="tab:purple", lw=2))
ax.annotate("", xy=X2.mean(0) + 2.5 * U2[:, 0], xytext=X2.mean(0), arrowprops=dict(arrowstyle="-|>", color="tab:red", lw=2))
ax.text(*(0.5 * L_unc * u_unc + [0.3, -0.3]), "uncentred 'PC1'", color="tab:purple", fontsize=8)
ax.text(*(X2.mean(0) + 2.5 * U2[:, 0] + [0.1, -0.5]), "true PC1", color="tab:red", fontsize=8)
ax.set_xlim(-0.5, X2[:, 0].max() + 1); ax.set_ylim(-0.5, X2[:, 1].max() + 1); ax.set_aspect("equal")
ax.set_xlabel("gene A (log-expression)"); ax.set_ylabel("gene B (log-expression)")
ax.set_title("Without centring, PC1 points at the mean,\nnot along the spread")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Without centring, $\frac1N X^\top X$ is *not* the covariance: it mixes the spread of the
data with the position of their mean. Here the mean $(3,9)$ is far from the origin, so the top eigenvector points
almost straight at the mean — it answers "where is the cloud?" rather than "in which direction does it spread?" — and
the variance of the cells along it is far below $\lambda_1$ (see the printout). **Centring moves the origin to the centre of the
cloud**, so that $X^\top X/N$ measures only variation. The same mean $\bar x$ must later be subtracted from any *new*
sample before projecting it.

Now we can write the whole recipe of slides 39–40 as one function, which we will reuse for the rest of the notebook.

In [ ]:
# ── 4.4 PCA in one function (slides 39–40) ──
def pca_eig(X, M):
    """PCA by eigendecomposition of the covariance matrix S = XᵀX/N of the centred data.

    Returns (mean, U_M, eigenvalues, scores): the column means, the K × M matrix of the top-M eigenvectors
    (sign convention: largest entry of each eigenvector positive), ALL K eigenvalues sorted from largest
    to smallest, and the N × M PC scores.
    """
    mean = X.mean(axis=0)
    Xc = X - mean                                       # 1. mean-centre
    S = Xc.T @ Xc / len(X)                              # 2. K × K covariance
    evals, evecs = np.linalg.eigh(S)                    # 3. eigendecomposition (ascending!)
    order = np.argsort(evals)[::-1]
    evals, evecs = np.clip(evals[order], 0, None), evecs[:, order]
    evecs = evecs * np.sign(evecs[np.argmax(np.abs(evecs), axis=0), np.arange(evecs.shape[1])])
    U_M = evecs[:, :M]                                  # 4. top-M principal directions
    return mean, U_M, evals, Xc @ U_M                   # 5. project: X_p = X [u1 … uM]


mean2, U2_check, lam2_check, _ = pca_eig(X2, 2)
sk = PCA(n_components=2).fit(X2)
print("same eigenvalues as before:", np.allclose(lam2_check, lam2))
print("scikit-learn agrees on the directions (|cosine| = 1):", np.abs(np.sum(sk.components_.T * U2_check, axis=0)).round(6))
print("scikit-learn's explained_variance_ divides by N−1:", sk.explained_variance_.round(4),
      "vs", (lam2 * N2 / (N2 - 1)).round(4))

### 📑 Slide connection — `011_PCA_mz_031825`, slides 32–40

- **What the slides show.** Step 2 projects data onto the PCs to get PC scores by inner products; the notes of slide 32 give $x^\top u_1 = \lVert x\rVert\cos\theta$ for a unit vector $u_1$. Slides 33–36 show a point $X$ and its projection $X_p$ on PC1, computed as $X^\top u_1$, and ask: 'Remember to mean center the data before projection (why?)'. Slides 37–38: $x=(2,1)$ means 'walk 2 steps along $u_1$'; with eigenvectors as axes you walk $X^\top u_1$ steps. Slides 39–40: the whole recipe and $X_p = X[u_1,\dots,u_M]$ ($N\times K$ times $K\times M$).
- **What this notebook adds.** An answer to the slide's 'why?' by doing it wrong: the uncentred top eigenvector points at the mean. Exact and rank-1 reconstructions, the identity 'reconstruction error = discarded eigenvalue', and a reusable `pca_eig` checked against scikit-learn (which divides by $N-1$).
- **Exam tip.** Know the shapes: $X$ is $N\times K$, $U_M$ is $K\times M$, $X_p$ is $N\times M$. A new sample $x$ is projected as $(x-\bar x)^\top U_M$ with the **training** mean $\bar x$.

---
## 5 · How many principal components?

> 📍 **Where we are.** With $K$ features there are $K$ eigenvalues and $K$ PCs (slide 43). Which $M$ should we keep?

The $k$-th eigenvalue is the variance of the data projected onto the $k$-th PC (slide 41), and they come sorted:
PC1 keeps the largest share of variation, PC2 the next largest, and so on (slide 42). The **fraction of variance
explained** by the first $M$ components is

$$\frac{\lambda_1 + \dots + \lambda_M}{\lambda_1 + \dots + \lambda_K} = \frac{\sum_{k=1}^{M}\lambda_k}{\mathrm{trace}(S)}.$$

*In words:* the share of the total spread of the data that survives the reduction. The slides' strategy (slide 43):
plot the eigenvalues — a **scree plot**, named after the rubble at the foot of a cliff — and keep the PCs with
reasonably high eigenvalues, i.e. those before the curve flattens. Slide 44 adds the alternative: choose $M$ to explain
a given percentage of the total variance.

We test this on a case where we *know* the answer: ten genes driven by **two** hidden regulatory programmes.

In [ ]:
# ── 5.1 Ten genes, two hidden programmes: a scree plot (SIMULATED) ──
rng = np.random.default_rng(SEED + 5)
N5, K5 = 200, 10
programmes = rng.normal(0, 1, (N5, 2))                    # two hidden programmes active in each sample
W_true = np.zeros((2, K5))
W_true[0, :5] = rng.uniform(0.7, 1.2, 5)                  # programme 1 drives genes 1–5
W_true[1, 5:] = rng.uniform(0.7, 1.2, 5)                  # programme 2 drives genes 6–10
X5 = programmes @ W_true + rng.normal(0, 0.4, (N5, K5))   # plus independent gene-level noise
mean5, U5, lam5, scores5 = pca_eig(X5, K5)
frac5 = lam5 / lam5.sum()
print(" PC   eigenvalue   % variance   cumulative %")
for k in range(K5):
    print(f"{k + 1:3d}   {lam5[k]:9.3f}   {100 * frac5[k]:9.1f}   {100 * frac5[:k + 1].sum():11.1f}")

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].bar(np.arange(1, K5 + 1), lam5, color="tab:blue")
axes[0].set_xlabel("principal component k"); axes[0].set_ylabel("eigenvalue λ_k (variance)")
axes[0].set_title("Scree plot: two big eigenvalues, then a flat floor")
axes[1].plot(np.arange(1, K5 + 1), 100 * np.cumsum(frac5), "o-", color="tab:green")
axes[1].axhline(90, color="0.5", ls="--"); axes[1].set_ylim(0, 102)
axes[1].set_xlabel("number of components M"); axes[1].set_ylabel("cumulative % variance explained")
axes[1].set_title("Cumulative variance (dashed: 90 %)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The scree plot drops sharply after PC2 and then stays on a flat "floor" of small, similar
eigenvalues — the noise, spread evenly over the remaining directions (its level is close to the noise variance
$0.4^2 = 0.16$). Two components explain most of the variance (see the cumulative column), exactly matching the two
hidden programmes we simulated. Real data are rarely this clean (you will see the Golub scree plot in Section 8), and
"keep 90 %" is a convention, not a law: how many components you need depends on what you want to do with them.

✏️ **Your turn.** Increase the gene-level noise from 0.4 to 1.2. Is the elbow still visible? How many PCs are now needed
for 90 % of the variance — and does that number still tell you how many *programmes* there are?

### 📑 Slide connection — `011_PCA_mz_031825`, slides 41–44

- **What the slides show.** The $k$-th eigenvalue is the variance of the data projected onto the $k$-th PC (slide 41); PC1 retains the greatest fraction of variation, PC $k$ the $k$-th greatest (slide 42); $K$ features give $K$ PCs — plot the eigenvalues and keep those that are reasonably high (slide 43). The PCA summary (slide 44) restates the recipe with $\lambda$'s sorted in descending order, calls $\sqrt{\lambda_k}u_k$ 'the $k$-th PC' and $u_k$ 'the $k$-th PC direction', defines PC scores $X_{\mathrm{proj},i}^\top = [X_i^\top u_1,\dots,X_i^\top u_K]$, shows $\mathrm{Var}(Xu_k)=u_k^\top S u_k=\lambda_k$, and suggests selecting the top $M$ PCs to explain a given percentage of the total variance $\sum_k\lambda_k$.
- **What this notebook adds.** A known-truth test (two programmes → two big eigenvalues) and the cumulative-variance curve. Terminology warning: 'principal component' is used for the direction $u_k$, for the scores $Xu_k$, and (slide 44) for the scaled vector $\sqrt{\lambda_k}u_k$; in this notebook 'PC $k$' means the direction $u_k$ and we say 'scores' for $Xu_k$. Slide 44's variance line writes $(X_i^\top u_k)^\top$ inside the sum where the square $(X_i^\top u_k)^2$ is meant (for a number, $a^\top=a$, so the intended formula is $\frac1N\sum_i (X_i^\top u_k)^2$).
- **Exam tip.** Fraction explained by PC $k$ = $\lambda_k/\sum_j\lambda_j$; total variance = trace$(S)$ = sum of the feature variances. A typical exam question gives eigenvalues and asks how many PCs reach 90 %.

---
## 6 · Why it works: PCA as an optimisation

> 📍 **Where we are.** We have *used* the fact that the top eigenvector is the direction of largest variance. The
> "additional material" of the slides (47–54) proves it. This derivation is a favourite exam question, so we go
> through it line by line and then check every step numerically.

**The problem (slide 47).** For centred $X$, the projection of sample $i$ on a direction $u$ is $x_i^\top u$, so the
variance of the projections is $\frac1N\sum_i (x_i^\top u)^2 = \frac1N u^\top X^\top X u = u^\top S u$. We want

$$\max_{u}\; u^\top S\, u \quad\text{subject to}\quad u^\top u = 1 .$$

*In words:* find the direction (a vector of length 1) along which the projected data vary the most. Without the
constraint we could make the variance as large as we like simply by making $u$ longer.

🧮 **Math refresher — Lagrange multipliers ([Module 04](04_calculus_and_optimization.ipynb)).** To maximise $f(u)$ subject to a constraint $g(u) = 0$, build
the **Lagrangian** $\mathcal L(u,\lambda) = f(u) + \lambda\, g(u)$ and set its derivative with respect to $u$ to zero.
At a constrained maximum, the "uphill" direction of $f$ must be perpendicular to the constraint surface, which is what
the extra term enforces; the number $\lambda$ is the **Lagrange multiplier**. We also need one rule of vector calculus:
for a symmetric matrix $A$, the gradient (vector of partial derivatives) of $u^\top A u$ is $2Au$, and that of
$u^\top u$ is $2u$.

**The derivation (slides 48–51).**

$$\mathcal L(u,\lambda) = u^\top S u + \lambda\,(1 - u^\top u)$$

*In words:* the variance we want to make large, plus $\lambda$ times "how badly the length-1 constraint is violated".

$$\frac{\partial \mathcal L}{\partial u} = 2Su - 2\lambda u = 0 \quad\Longrightarrow\quad S u = \lambda u .$$

*In words:* at the best direction, multiplying by $S$ only rescales $u$ — the best direction **must be an
eigenvector** of $S$, and the Lagrange multiplier **is its eigenvalue**. (The slides write $(1/N)X^\top Xu - \lambda u = 0$,
having divided the factor 2 out.) Which eigenvector? Plug $Su = \lambda u$ back into the objective:

$$u^\top S u = u^\top(\lambda u) = \lambda\, u^\top u = \lambda .$$

*In words:* the variance along an eigenvector equals its eigenvalue, so the **largest** eigenvalue gives the maximum:
$u_1$ is the first PC and $\lambda_1$ its variance.

**Proof that nothing beats $u_1$ (slide 52).** Any unit vector can be written in the eigenvector basis,
$u = \sum_k a_k u_k$ with $\sum_k a_k^2 = 1$. Then $u^\top S u = \sum_k a_k^2 \lambda_k \le \lambda_1 \sum_k a_k^2 = \lambda_1$.
*In words:* the variance in any direction is a weighted average of the eigenvalues, and an average can never exceed
its largest member. **Later PCs (slide 53):** $u_2$ maximises the variance among directions perpendicular to $u_1$, and so on.

In [ ]:
# ── 6.1 Brute force in 10-D: no random direction beats λ1 ──
rng = np.random.default_rng(SEED + 6)
X5c = X5 - mean5
S5 = X5c.T @ X5c / N5
u_rand = rng.normal(size=(20000, K5))
u_rand /= np.linalg.norm(u_rand, axis=1, keepdims=True)       # 20,000 random unit vectors in 10-D
var_rand = np.einsum("ij,jk,ik->i", u_rand, S5, u_rand)        # uᵀ S u for every random u
print(f"largest variance among 20,000 random directions: {var_rand.max():.3f}   λ1  = {lam5[0]:.3f}")
print(f"smallest variance among them:                     {var_rand.min():.3f}   λ10 = {lam5[-1]:.3f}")

fig, ax = plt.subplots(figsize=(6.5, 3))
ax.hist(var_rand, bins=80, color="0.6")
ax.axvline(lam5[0], color="tab:red", lw=2, label="λ1 (top eigenvalue)")
ax.axvline(lam5[-1], color="tab:blue", lw=2, label="λ10 (smallest)")
ax.set_xlabel("variance of the data projected on a random unit vector u  (uᵀSu)"); ax.set_ylabel("count")
ax.set_title("Every direction's variance lies between the smallest and the largest eigenvalue"); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# ── 6.2 The proof of slide 52, the Lagrangian condition and the 2nd PC, numerically ──
U5_all = pca_eig(X5, K5)[1]                               # all 10 eigenvectors (columns)
u = u_rand[0]
a = U5_all.T @ u                                          # coordinates of u in the eigenvector basis
print(f"Σ a_k² = {np.sum(a ** 2):.6f}   uᵀSu = {u @ S5 @ u:.4f}   Σ a_k² λ_k = {np.sum(a ** 2 * lam5):.4f}   ≤ λ1 = {lam5[0]:.4f}")


def lagrangian_gradient(u, S):
    """Gradient of uᵀSu + λ(1 − uᵀu) with respect to u, at the best multiplier λ = uᵀSu."""
    lam = u @ S @ u
    return 2 * S @ u - 2 * lam * u


print("‖∇L‖ at u1 (an eigenvector):", np.linalg.norm(lagrangian_gradient(U5_all[:, 0], S5)).round(12),
      "| at a random unit vector:", np.linalg.norm(lagrangian_gradient(u, S5)).round(3))

# Slide 53: among directions perpendicular to u1, the best one has variance λ2
u_perp = u_rand - np.outer(u_rand @ U5_all[:, 0], U5_all[:, 0])     # remove the u1 component
u_perp /= np.linalg.norm(u_perp, axis=1, keepdims=True)
var_perp = np.einsum("ij,jk,ik->i", u_perp, S5, u_perp)
print(f"best variance among random directions ⟂ u1: {var_perp.max():.3f}   λ2 = {lam5[1]:.3f}")

🔍 **What just happened?** Among 20,000 random directions in 10-D, none had a variance above $\lambda_1$ or below
$\lambda_{10}$ — the eigenvalues are the hard ceiling and floor, exactly as the proof says. The proof itself checks out
numerically: the squared coordinates $a_k^2$ of a unit vector add up to 1, and $u^\top S u$ equals the weighted average
$\sum_k a_k^2\lambda_k$. The Lagrangian's gradient vanishes at an eigenvector and not elsewhere; and when we restrict
the search to directions perpendicular to $u_1$, the ceiling drops to $\lambda_2$.

Notice also that random search *never quite reaches* $\lambda_1$ in 10-D: most random directions mix many eigenvectors.
In 3,571 dimensions random search would be hopeless — the eigendecomposition is not just elegant, it is necessary.

✏️ **Your turn.** A simple algorithm finds $u_1$ without `eigh`: start from a random unit vector and repeat
$u \leftarrow Su/\lVert Su\rVert$ (**power iteration**). Implement it in three lines and check after 50 iterations that
$u^\top S u \approx \lambda_1$. Why does it work? (Hint: write $u$ in the eigenvector basis; each multiplication by $S$
multiplies coordinate $k$ by $\lambda_k$.)

In [ ]:
# ── 6.3 PCs are orthogonal and their scores are uncorrelated ──
corr_genes = np.corrcoef(X5, rowvar=False)
cov_scores = scores5.T @ scores5 / N5                     # covariance matrix of the 10 PC scores
off = ~np.eye(K5, dtype=bool)
print(f"largest |correlation| between two original genes: {np.abs(corr_genes[off]).max():.3f}")
print(f"largest |covariance| between two PC scores:       {np.abs(cov_scores[off]).max():.1e}")
print("diagonal of the score covariance = eigenvalues:", np.allclose(np.diag(cov_scores), lam5))
print("UᵀU = I:", np.allclose(U5_all.T @ U5_all, np.eye(K5)))

🔍 **What just happened?** The original genes are strongly correlated (genes driven by the same programme carry
redundant information), but the PC scores are **uncorrelated** — their covariance matrix is diagonal, with the
eigenvalues on the diagonal ($U^\top S U = D$). This is slide 54's point: PCA removes the correlated (redundant)
dimensions and describes the data with orthogonal axes.

### 6.4 The SVD route — and why it matters when $K \gg N$

🧮 **Math refresher — the singular value decomposition (SVD).** Any $N\times K$ matrix can be factored into an
$N\times r$ matrix $V$ with orthonormal columns, a diagonal matrix $\Sigma$ of **singular values**
$\sigma_1\ge\sigma_2\ge\dots\ge 0$, and the transpose of a $K\times r$ matrix $W$ with orthonormal columns (we use
$V, W$ so as not to clash with the slides' $U$). Because $V^\top V = I$, the covariance follows immediately:

$$X = V\,\Sigma\,W^\top \quad\Longrightarrow\quad S = \frac1N X^\top X = W\,\frac{\Sigma^2}{N}\,W^\top .$$

*In words:* the second formula is an eigendecomposition of $S$, so the right singular vectors $W$ **are** the principal directions, the eigenvalues are $\lambda_k=\sigma_k^2/N$,
and the scores are $XW = V\Sigma$ — all without ever forming the $K\times K$ matrix $S$. For the Golub data $S$ would be
$3{,}571\times3{,}571$ (12.8 million numbers) although only $N=72$ samples went into it.

In [ ]:
# ── 6.4 eigh of the 3,571 × 3,571 covariance vs SVD of the 72 × 3,571 data matrix ──
Xg_c = X_golub - X_golub.mean(axis=0)
t0 = time.perf_counter()
S_g = Xg_c.T @ Xg_c / N                                   # K × K
ev_g, evec_g = np.linalg.eigh(S_g)
t_eig = time.perf_counter() - t0
t0 = time.perf_counter()
V_g, sig_g, Wt_g = np.linalg.svd(Xg_c, full_matrices=False)   # V: N × N, sig: N, Wᵀ: N × K
t_svd = time.perf_counter() - t0
print(f"eigh on S ({S_g.shape[0]:,} × {S_g.shape[1]:,}): {t_eig:.2f} s   |   SVD of X {Xg_c.shape}: {t_svd:.3f} s")
print("top 5 eigenvalues from eigh:   ", ev_g[::-1][:5].round(4))
print("top 5 from SVD, σ²/N:          ", (sig_g[:5] ** 2 / N).round(4))
print("same PC1 direction (|cosine|):  ", abs(evec_g[:, -1] @ Wt_g[0]).round(8))
print(f"non-zero eigenvalues of S: {np.sum(ev_g > 1e-10 * ev_g.max())} — at most N − 1 = {N - 1} after centring")
del S_g, evec_g                                           # free ~100 MB of memory

🔍 **What just happened?** Both routes give identical eigenvalues and the same PC1, but the SVD of the thin
$72\times3{,}571$ matrix is far faster than eigendecomposing the huge $3{,}571\times3{,}571$ covariance (see the
timings). And only 71 of the 3,571 eigenvalues are non-zero: 72 centred points span at most a 71-dimensional space, so
at most $N-1$ PCs carry any variance. This is why genetics software computes PCs of $10^5$–$10^6$ SNPs by the SVD (or
by eigendecomposing the small $N\times N$ matrix $XX^\top$), never by building $S$.

### 📑 Slide connection — `011_PCA_mz_031825`, slides 47–54

- **What the slides show.** PCA as optimisation: maximise $(1/N)u^\top X^\top Xu$ s.t. $u^\top u=1$ (slide 47); Lagrangian $(1/N)u^\top X^\top Xu + \lambda(1-u^\top u)$ (slide 48); differentiate and set to zero: $(1/N)X^\top Xu-\lambda u=0$, i.e. $Su=\lambda u$, so $u$ must be an eigenvector (slides 49–51). Slide 52 proves $u_1^\top Su_1=\lambda_1$ is the largest possible value by expanding $u=\sum_k a_ku_k$; slide 53 adds that $u_2$ maximises the variance among directions orthogonal to $u_1$, and so on. Slide 54: PCA preserves variability, removes correlated (redundant) dimensions, and PCs are orthogonal.
- **What this notebook adds.** The factor 2 of the derivative (which the slides divide out), the 'plug back in' step showing variance = eigenvalue, brute-force checks with 20,000 random directions (ceiling $\lambda_1$, floor $\lambda_K$, ceiling $\lambda_2$ when $\perp u_1$), decorrelation of the scores, power iteration, and the SVD route with a real timing on Golub. Small correction: slide 52 writes a strict '<' in $\sum_k a_k^2\lambda_k < \lambda_1\sum_k a_k^2$; it should be '≤' (equality holds for $u=-u_1$, or whenever $\lambda_1=\lambda_2$).
- **Exam tip.** Reproduce the four lines: objective + constraint → Lagrangian → gradient $2Su-2\lambda u=0$ → plug back: variance $=\lambda$, so take the largest. Know why the SVD is preferred when $K\gg N$ and that at most $\min(N-1, K)$ eigenvalues are non-zero.

---
## 7 · Genes mirror geography: a simulated stepping-stone population

> 📍 **Where we are.** We now have a complete, verified PCA. Time for the flagship example of the slides (slides 4–5 and
> 45): genotype PCA recovering a map. We cannot ship the Novembre et al. data, so we **simulate** a population in which
> we *know* where everybody lives, and check whether PCA can find it out.

🧫 **Isolation by distance.** Imagine 40 villages (**demes**) on an $8\times5$ grid. Each generation, a small fraction
$m$ of every village's gene pool comes from its four neighbours (migration), and every village's allele frequencies
drift randomly because only a finite number of gene copies ($2N_e$) are passed on (**genetic drift**, a binomial
sampling step). This is Kimura's **stepping-stone model** (Kimura & Weiss 1964). Drift makes villages different;
migration keeps *neighbours* similar. After many generations, allele frequencies vary smoothly across the map, and
two individuals are genetically more similar the closer they live — **isolation by distance**.

Genotypes are coded 0/1/2 (copies of one allele, as everywhere in the course). Before PCA we standardise each SNP $j$
with its allele frequency $\hat p_j$, as is standard in genetics (Patterson, Price & Reich 2006):

$$z_{ij} = \frac{g_{ij} - 2\hat p_j}{\sqrt{2\hat p_j(1-\hat p_j)}}$$

*In words:* subtract the expected genotype under Hardy–Weinberg ($2\hat p_j$) and divide by its expected standard
deviation, so that rare and common SNPs contribute comparably. (Patterson *et al.* divide by $\sqrt{\hat p_j(1-\hat p_j)}$;
the constant factor $\sqrt2$ rescales every SNP equally and leaves the PCs unchanged.)

In [ ]:
# ── 7.1 Simulate allele frequencies on a grid of demes (SIMULATED stepping-stone model) ──
rng = np.random.default_rng(SEED + 7)
NX, NY = 8, 5                                             # 8 × 5 = 40 demes on a map
deme_xy = np.array([(i, j) for j in range(NY) for i in range(NX)], float)
n_demes = len(deme_xy)
steps_apart = np.abs(deme_xy[:, None, :] - deme_xy[None, :, :]).sum(-1)    # grid distance between demes
m_rate = 0.10                                             # fraction of gene copies coming from neighbours
MIG = (steps_apart == 1) / (steps_apart == 1).sum(axis=1, keepdims=True) * m_rate
np.fill_diagonal(MIG, 1 - m_rate)                         # each row sums to 1
K_snps, Ne, n_gen = 3000, 100, 150
P_freq = np.tile(rng.uniform(0.2, 0.8, K_snps), (n_demes, 1))  # all demes start with the same frequencies
for _ in range(n_gen):
    P_freq = MIG @ P_freq                                 # migration: mix with the neighbours
    P_freq = rng.binomial(2 * Ne, P_freq) / (2 * Ne)      # drift: 2·Ne gene copies sampled at random
p_bar = P_freq.mean(axis=0)
fst = np.mean(P_freq.var(axis=0)) / np.mean(p_bar * (1 - p_bar))
print(f"{n_demes} demes × {K_snps:,} SNPs after {n_gen} generations;  F_ST ≈ {fst:.3f} "
      f"(fraction of genetic variance between demes)")

In [ ]:
# ── 7.2 Sample individuals; two allele-frequency maps ──
n_per = 12                                                # individuals sampled per deme → N = 480
deme_of = np.repeat(np.arange(n_demes), n_per)
G_geo = rng.binomial(2, P_freq[deme_of]).astype(float)    # N × K genotype matrix, 0/1/2
p_hat = G_geo.mean(axis=0) / 2
keep = (p_hat > 0.02) & (p_hat < 0.98)                    # drop SNPs that are (almost) fixed in the sample
G_geo, p_hat = G_geo[:, keep], p_hat[keep]
xy_true = deme_xy[deme_of] + rng.uniform(-0.3, 0.3, (len(deme_of), 2))   # jittered home of each person
print("genotype matrix:", G_geo.shape, "(individuals × SNPs)")

spread = P_freq[:, keep].std(axis=0)
fig, axes = plt.subplots(1, 2, figsize=(9, 2.8))
for ax, j in zip(axes, np.argsort(spread)[::-1][[0, 5]]):
    im = ax.imshow(P_freq[:, keep][:, j].reshape(NY, NX), origin="lower", cmap="viridis")
    ax.set_title(f"SIMULATED allele frequency of SNP {j} across the map")
    ax.set_xlabel("east–west position (deme)"); ax.set_ylabel("south–north")
    plt.colorbar(im, ax=ax, fraction=0.03)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Each SNP's allele frequency varies in smooth patches across the map: neighbouring demes have
similar frequencies because they exchange migrants, while distant demes have drifted apart. The overall differentiation
($F_{ST}$, printed above) is small: about 95 % of the genetic variance lies *within* demes. Humans are similar — even
continental groups differ by an $F_{ST}$ of only very roughly 0.1, and populations within a continent by much less, so
our simulated map is, if anything, easier to recover than the real European one. No single SNP tells you where someone
lives. The question is whether thousands of weak,
smoothly varying signals add up to a map.

In [ ]:
# ── 7.3 PCA of the standardised genotype matrix ──
Z_geo = (G_geo - 2 * p_hat) / np.sqrt(2 * p_hat * (1 - p_hat))   # Patterson et al. (2006) scaling
V_geo, sig_geo, Wt_geo = np.linalg.svd(Z_geo - Z_geo.mean(axis=0), full_matrices=False)
lam_geo = sig_geo ** 2 / len(Z_geo)
pc_geo = V_geo[:, :2] * sig_geo[:2]                      # scores on PC1 and PC2
print("% variance explained by PC1–PC6:", np.round(100 * lam_geo[:6] / lam_geo.sum(), 2))


def procrustes_rotate(A, B):
    """Rotate/reflect and scale the 2-D point cloud A to best match B in the least-squares sense.

    The best orthogonal matrix comes from the SVD of A0ᵀB0 (the 'orthogonal Procrustes' problem).
    """
    A0, B0 = A - A.mean(axis=0), B - B.mean(axis=0)
    L, s, Rt = np.linalg.svd(A0.T @ B0)
    R = L @ Rt
    return (s.sum() / np.sum(A0 ** 2)) * A0 @ R + B.mean(axis=0), R


home = deme_xy[deme_of]
pc_map, R_geo = procrustes_rotate(pc_geo, home)
r_x = np.corrcoef(pc_map[:, 0], home[:, 0])[0, 1]
r_y = np.corrcoef(pc_map[:, 1], home[:, 1])[0, 1]
print(f"after rotation: corr(east–west, true) = {r_x:.3f}   corr(south–north, true) = {r_y:.3f}")
print(f"rotation angle ≈ {np.degrees(np.arctan2(R_geo[1, 0], R_geo[0, 0])):.0f}° "
      f"(det = {np.linalg.det(R_geo):+.0f}: −1 means a mirror image was needed)")

In [ ]:
# ── 7.4 The PC map next to the real map ──
colour = np.column_stack([home[:, 0] / (NX - 1), 0.25 + 0.5 * home[:, 1] / (NY - 1),
                          1 - home[:, 0] / (NX - 1)])        # colour = east–west hue, brightness = north
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
axes[0].scatter(xy_true[:, 0], xy_true[:, 1], c=colour, s=10)
axes[0].set_title("true homes (SIMULATED grid of 40 demes)")
axes[0].set_xlabel("east–west"); axes[0].set_ylabel("south–north")
axes[1].scatter(pc_geo[:, 0], pc_geo[:, 1], c=colour, s=10)
axes[1].set_title("PC1 vs PC2 of 480 genomes"); axes[1].set_xlabel("PC1 score"); axes[1].set_ylabel("PC2 score")
axes[2].scatter(pc_map[:, 0], pc_map[:, 1], c=colour, s=10)
axes[2].set_title(f"PCs after Procrustes rotation\n(r = {r_x:.2f} east–west, {r_y:.2f} south–north)")
axes[2].set_xlabel("rotated PC coordinate 1"); axes[2].set_ylabel("rotated PC coordinate 2")
for ax in axes:
    ax.set_aspect("equal", adjustable="datalim")
plt.tight_layout(); plt.show()

🔍 **What just happened?** PCA was given only a $480 \times K$ table of 0/1/2 genotypes — no coordinates, no village
labels. Yet the first two PCs arrange the individuals in the layout of the grid: colours that are neighbours on the
real map are neighbours in the PC plot. PC1 runs along the long (east–west) axis of the grid, which carries the most
genetic variance, and PC2 along the short one. After a rotation (PCA cannot know which way is north, and a PC's sign
is arbitrary) — the *orthogonal Procrustes* fit, i.e. the rotation (possibly with a mirror flip) and overall scaling
that best superimpose the PC cloud on the true map — the PC coordinates correlate strongly with the true positions
(see the printed correlations). This is
the mechanism behind Novembre et al.'s map of Europe. The percentages of variance explained are tiny (about 2 % for
PC1 here): **a PC can carry very little of the total variance and still carry a very clear signal**, because it
collects thousands of weak, consistent effects.

⚠️ Before believing any such picture, run a **negative control**: destroy the geography and check that the map
disappears.

In [ ]:
# ── 7.5 Negative control: shuffle each SNP across individuals ──
G_shuf = rng.permuted(G_geo, axis=0)                      # each column shuffled independently → no structure
Z_shuf = (G_shuf - 2 * p_hat) / np.sqrt(2 * p_hat * (1 - p_hat))
V_s, sig_s, _ = np.linalg.svd(Z_shuf - Z_shuf.mean(axis=0), full_matrices=False)
lam_shuf = sig_s ** 2 / len(Z_shuf)
pc_shuf_map, _ = procrustes_rotate(V_s[:, :2] * sig_s[:2], home)
print("% variance PC1–PC6, shuffled:", np.round(100 * lam_shuf[:6] / lam_shuf.sum(), 2))
print(f"after rotation: corr with east–west = {np.corrcoef(pc_shuf_map[:, 0], home[:, 0])[0, 1]:.3f}, "
      f"south–north = {np.corrcoef(pc_shuf_map[:, 1], home[:, 1])[0, 1]:.3f}")

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(np.arange(1, 21), 100 * lam_geo[:20] / lam_geo.sum(), "o-", color="tab:blue", label="isolation by distance")
ax.plot(np.arange(1, 21), 100 * lam_shuf[:20] / lam_shuf.sum(), "s-", color="0.5", label="shuffled (no structure)")
ax.set_xlabel("principal component k"); ax.set_ylabel("% variance explained")
ax.set_title("Structure shows up as eigenvalues standing out of the noise floor"); ax.legend()
plt.tight_layout(); plt.show()

🔍 **What just happened?** With the geography destroyed, the eigenvalues form a smooth, slowly declining noise floor
(even pure noise has a "largest" eigenvalue — the leading ones are always somewhat above average), and the rotated PCs
no longer correlate with where people live. With real structure, the first few eigenvalues **stand out** above that
floor. Comparing a scree plot with a null scree plot is a simple, honest way to decide which PCs reflect structure.

✏️ **Your turn.** (a) Set `m_rate = 0.5` (lots of migration) and re-run 7.1–7.4. What happens to $F_{ST}$ and to the
map? (b) Set `NX, NY = 6, 6` (a square). Which axis does PC1 pick now — and why can the answer change between seeds?
(c) Plot PC3 and PC4 scores as colours on the true map: Novembre & Stephens (2008) showed that higher PCs of spatial
data look like "waves" (sinusoids) over the map. Do you see them?

### 📑 Slide connection — `011_PCA_mz_031825`, slides 4–5, 45

- **What the slides show.** Genotype data with 200K SNPs reduced to two linearly transformed features capturing the largest variation (slide 4). Novembre et al. 2008 Nature: 197,146 SNPs, 1,387 European individuals, reduced from 197,146 to 2 dimensions by PCA; each dot is an individual represented by the top two PCs, and the scatter resembles the map of Europe (slides 5 and 45; 'more in 02-744').
- **What this notebook adds.** A mechanistic, fully SIMULATED analogue where the truth is known: a stepping-stone model with migration and drift, Patterson-style SNP standardisation, PCA by SVD, a Procrustes rotation onto the true map with correlations, and a shuffled negative control. It does **not** reproduce the paper's data or numbers; real analyses also face uneven sampling, admixture and linkage between SNPs.
- **Exam tip.** PCs reflect the dominant axes of genetic variance *in the sample analysed*: with isolation by distance they track geography, but sample composition (who was genotyped, how many from where) shapes them too. A PC explaining 1–2 % of variance can still be highly informative.

---
## 8 · Real leukaemia expression: what does PCA see in the Golub data?

> 📍 **Where we are.** Module 12 found clusters in the Golub patients; Module 00 showed a PCA plot as a black box. Now
> you can open the box. We run PCA on all 72 patients **without labels** and colour the points by ALL/AML only
> *afterwards* — the labels play no part in the computation.

We use the SVD route (Section 6.4). Two preprocessing choices are common for expression data: **centring only** (genes
keep their natural spread on the log scale, so highly variable genes weigh more) and **standardising** (every gene
scaled to variance 1, so all genes weigh equally). We try both.

In [ ]:
# ── 8.1 PCA of the Golub data, genes centred (not scaled) ──
def class_separation(score, y):
    """Standardised difference of the mean score between the two classes (AML − ALL)."""
    a, b = score[y == 0], score[y == 1]
    return (b.mean() - a.mean()) / np.sqrt((a.var() + b.var()) / 2)


def svd_pca(X):
    """PCA by SVD of the centred data: returns (eigenvalues, N × r scores, r × K directions), signs fixed."""
    Xc = X - X.mean(axis=0)
    V, sig, Wt = np.linalg.svd(Xc, full_matrices=False)
    flip = np.sign(Wt[np.arange(len(Wt)), np.argmax(np.abs(Wt), axis=1)])   # largest loading positive
    return sig ** 2 / len(X), V * sig * flip, Wt * flip[:, None]


lam_raw, sc_raw, W_raw = svd_pca(X_golub)
print("% variance, PC1–PC5:", np.round(100 * lam_raw[:5] / lam_raw.sum(), 1),
      f"| PCs needed for 90 %: {np.searchsorted(np.cumsum(lam_raw) / lam_raw.sum(), 0.9) + 1}")
for k in range(4):
    print(f"PC{k + 1}: AML-vs-ALL separation = {class_separation(sc_raw[:, k], y_golub):+.2f} SD")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].plot(np.arange(1, 31), 100 * lam_raw[:30] / lam_raw.sum(), "o-", color="tab:blue", ms=4)
axes[0].set_xlabel("principal component k"); axes[0].set_ylabel("% variance explained")
axes[0].set_title("Golub scree plot: no clean elbow")
for c, lab in enumerate(["ALL", "AML"]):
    m = y_golub == c
    axes[1].scatter(sc_raw[m, 0], sc_raw[m, 1], s=18, color=CLASS_COLORS[c], edgecolor="k", lw=0.3, label=lab)
axes[1].set_xlabel("PC1 score"); axes[1].set_ylabel("PC2 score"); axes[1].legend(title="coloured afterwards")
axes[1].set_title(f"{golub['source']}: genes centred")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Unlike the toy of Section 5, the scree plot of real expression has no clean elbow: variance
decays gradually over many components, and dozens of PCs are needed to reach 90 % (see the printout). Yet the very
first PC, computed without the labels, separates ALL from AML well (the largest standardised separation in the
printout, for PC1): the biggest single axis of variation among these patients' genes is the leukaemia type. That is
"class discovery" — the question in the title of Golub et al. (1999) — seen through PCA.

Now the other preprocessing choice.

In [ ]:
# ── 8.2 Standardised genes: a different PC1 — and what it tracks ──
lam_std, sc_std, W_std = svd_pca(Z_golub)                 # Z_golub = z-scored genes (Section 1.2)
sample_mean = X_golub.mean(axis=1)                        # each array's average log-expression
print("% variance, PC1–PC5:", np.round(100 * lam_std[:5] / lam_std.sum(), 1))
for k in range(3):
    print(f"PC{k + 1}: AML-vs-ALL separation {class_separation(sc_std[:, k], y_golub):+.2f} SD | "
          f"correlation with the array's mean expression {np.corrcoef(sc_std[:, k], sample_mean)[0, 1]:+.2f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
for c, lab in enumerate(["ALL", "AML"]):
    m = y_golub == c
    axes[0].scatter(sc_std[m, 0], sc_std[m, 1], s=18, color=CLASS_COLORS[c], edgecolor="k", lw=0.3, label=lab)
    axes[1].scatter(sample_mean[m], sc_std[m, 0], s=18, color=CLASS_COLORS[c], edgecolor="k", lw=0.3, label=lab)
axes[0].set_xlabel("PC1 score"); axes[0].set_ylabel("PC2 score"); axes[0].legend()
axes[0].set_title("Same data, genes standardised")
axes[1].set_xlabel("array's mean log10 expression (over all genes)"); axes[1].set_ylabel("PC1 score (standardised)")
axes[1].set_title("Standardised PC1 ≈ overall array brightness")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Standardising changed the story. Now PC1 barely separates ALL from AML; instead it is almost
perfectly correlated with each array's **average expression over all genes** (correlation shown in the printout) — an
overall "brightness" of the microarray that varies from sample to sample for technical reasons (amount of RNA,
hybridisation, scanner). Giving every gene equal weight lets thousands of low-variance genes that share this global
shift dominate PC1, and the leukaemia signal moves to PC2 and PC3. Neither choice is "the correct PCA": they answer
different questions, and the first PC is simply the largest *source of variance* — biological or not. Array
normalisation (for example subtracting each sample's mean) is designed to remove exactly this kind of effect.

✏️ **Your turn.** Remove the brightness effect *before* standardising: `Xn = X_golub - X_golub.mean(axis=1, keepdims=True)`,
then run `svd_pca(standardize(Xn)[0])` and print the separation and the brightness correlation of PC1–PC3. What changes?

In [ ]:
# ── 8.3 Loadings: which genes drive the (centred) PC1? ──
top = np.argsort(-np.abs(W_raw[0]))[:12]                  # genes with the largest |weight| in PC1
print(f"PC1 score is higher in {'AML' if class_separation(sc_raw[:, 0], y_golub) > 0 else 'ALL'} patients.")
print(f"{'probe ID':>16s} {'loading':>8s} {'mean ALL':>9s} {'mean AML':>9s}")
for j in top:
    print(f"{golub['genes'][j]:>16s} {W_raw[0, j]:+8.3f} {X_golub[y_golub == 0, j].mean():9.2f} "
          f"{X_golub[y_golub == 1, j].mean():9.2f}")

fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(range(len(top)), W_raw[0, top], color=["tab:red" if w > 0 else "tab:blue" for w in W_raw[0, top]])
ax.set_xticks(range(len(top)), golub["genes"][top], rotation=60, ha="right", fontsize=7)
ax.set_ylabel("loading on PC1 (entry of u1)")
ax.set_title("The 12 genes with the largest PC1 loadings (red: higher PC1 score ↔ higher expression)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The **loadings** (the entries of $u_1$) tell you how much each gene contributes to the PC1
score. Genes with large positive loadings are higher in the patients with high PC1 scores, genes with negative loadings
lower — compare the class means in the table. Loadings turn a PCA plot into biological hypotheses ("these genes
distinguish the two leukaemias"), which you would then test on new samples or look up: the IDs are Affymetrix HU6800
probe identifiers, mostly GenBank accession numbers.

⚠️ **Caveats.** (1) Each loading is tiny (about 0.1) because $u_1$ has length 1 across 3,571 genes — PC1 is a
*collective* pattern, not a few genes. (2) A gene's loading reflects its variance *and* its correlation with the
pattern, so with centred data highly variable genes dominate. (3) This PCA used all 72 patients; if you later use PC
scores as features in a classifier, the PCA must be fitted inside each training fold ([Module 23](23_model_selection.ipynb)'s selection-bias lesson).

---
## 9 · Images: eigen-digits and reconstruction from four numbers

> 📍 **Where we are.** The slides finish with images (slides 55–58): every image is a long vector of pixel values, so a
> collection of images is again an $N\times K$ matrix, and its principal components are themselves **images**.

The slides use face photographs (each a vector with 83,781 entries); we use the 1,797 handwritten digits that ship with
scikit-learn: $8\times8$ grey-level images, i.e. $K = 64$ features. Microscopy images of cells or tissues are treated
the same way. An image $x$ is represented by its $M$ PC coefficients $q_k = (x - \bar x)^\top u_k$ (the slides'
$[q_1, q_2, q_3, q_4]$) and reconstructed as

$$\hat x = \bar x + \sum_{k=1}^{M} q_k\, u_k .$$

*In words:* start from the average image and add a weighted sum of "eigen-images"; the weights are the coefficients.

In [ ]:
# ── 9.1 Eigen-digits: the principal components are images ──
digits = load_digits()
X_dig, y_dig = digits.data, digits.target                  # 1,797 images × 64 pixels (grey levels 0–16)
mean_dig, U_dig, lam_dig, _ = pca_eig(X_dig, 64)
print("data:", X_dig.shape, "| % variance PC1–PC8:", np.round(100 * lam_dig[:8] / lam_dig.sum(), 1))

fig, axes = plt.subplots(1, 9, figsize=(12, 1.9))
axes[0].imshow(mean_dig.reshape(8, 8), cmap="gray_r"); axes[0].set_title("mean image", fontsize=8)
for k in range(8):
    lim = np.abs(U_dig[:, k]).max()
    axes[k + 1].imshow(U_dig[:, k].reshape(8, 8), cmap="RdBu_r", vmin=-lim, vmax=lim)
    axes[k + 1].set_title(f"PC{k + 1} ({100 * lam_dig[k] / lam_dig.sum():.0f} %)", fontsize=8)
for ax in axes:
    ax.axis("off")
fig.suptitle("Eigen-digits (red: add ink, blue: remove ink)", fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# ── 9.2 Reconstruct one digit from M = 1, 2, 4, … coefficients ──
i_show = int(np.flatnonzero(y_dig == 3)[0])               # the first "3" in the data set
x0 = X_dig[i_show]
q = (x0 - mean_dig) @ U_dig                               # all 64 coefficients of this image
print("[q1, q2, q3, q4] =", q[:4].round(2), " ← the slide's 4-dimensional representation")
Ms = [0, 1, 2, 4, 8, 16, 32, 64]
fig, axes = plt.subplots(1, len(Ms) + 1, figsize=(12, 1.9))
axes[0].imshow(x0.reshape(8, 8), cmap="gray_r"); axes[0].set_title("original\n(64 numbers)", fontsize=8)
for ax, M in zip(axes[1:], Ms):
    x_hat = mean_dig + U_dig[:, :M] @ q[:M]
    ax.imshow(x_hat.reshape(8, 8), cmap="gray_r", vmin=0, vmax=16)
    ax.set_title(f"M = {M}\nerror {np.sum((x0 - x_hat) ** 2):.0f}", fontsize=8)
for ax in axes:
    ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# ── 9.3 Average reconstruction error = sum of the discarded eigenvalues ──
Xd_c = X_dig - mean_dig
err = []
for M in range(65):
    X_hat = Xd_c @ U_dig[:, :M] @ U_dig[:, :M].T          # project onto M PCs and map back
    err.append(np.mean(np.sum((Xd_c - X_hat) ** 2, axis=1)))
err = np.array(err)
discarded = np.array([lam_dig[M:].sum() for M in range(65)])
print("error = Σ discarded λ for every M:", np.allclose(err, discarded))
cum = np.cumsum(lam_dig) / lam_dig.sum()
print(f"M for 50 % / 90 % / 95 % of the variance: {np.searchsorted(cum, 0.5) + 1} / "
      f"{np.searchsorted(cum, 0.9) + 1} / {np.searchsorted(cum, 0.95) + 1} (of 64)")

_, _, _, sc_dig = pca_eig(X_dig, 2)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].plot(range(65), err, color="tab:purple")
axes[0].set_xlabel("number of components M"); axes[0].set_ylabel("mean squared reconstruction error")
axes[0].set_title("Error falls fast, then slowly")
sca = axes[1].scatter(sc_dig[:, 0], sc_dig[:, 1], c=y_dig, cmap="tab10", s=5)
axes[1].set_xlabel("PC1 score"); axes[1].set_ylabel("PC2 score"); axes[1].set_title("All 1,797 digits on 2 PCs")
plt.colorbar(sca, ax=axes[1], label="digit (coloured afterwards)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The first eigen-digits are smooth patterns of "add ink here, remove ink there" — PC1, for
example, contrasts strokes typical of some digits with those of others. Reconstructing the "3" from only four numbers
gives a blurry blob in which the outline of the digit starts to appear (its squared error is about half that of the
mean image alone, $M = 0$); the error shrinks steadily as $M$ grows, and by $M = 16$ the "3" is clearly recognisable. Averaged over all images the
reconstruction error equals **exactly** the sum of the discarded eigenvalues (checked for every $M$): PCA is the best
linear compression in the squared-error sense, and the eigenvalues tell you its price in advance. In two dimensions the
digit classes overlap but are far from randomly mixed — PCA is a good first look, not a classifier.

✏️ **Your turn.** Pick a "1" (`y_dig == 1`) instead of a "3". How many components does it need to be recognisable?
Then reconstruct a digit using only PCs 5–8 (no PC1–4). What does that show about what the leading PCs encode?

### 📑 Slide connection — `011_PCA_mz_031825`, slides 55–59

- **What the slides show.** Applying PCA to images: face photographs of '361 × 261 pixels, 83781 dimensional data' (slide 55); dimensionality reduction from $K$ to $M<K$ (slide 56); the principal components are images (slide 57); projecting a face onto the four PCs gives a 4-dimensional representation $[q_1,q_2,q_3,q_4]$ (slide 58). Summary (slide 59): PCA is unsupervised; it reduces dimension while preserving variability; in practice use the eigendecomposition; top eigenvalues = dimensions with the most variability, their eigenvectors = the new axes.
- **What this notebook adds.** The same idea on 64-pixel digits with code: eigen-digits, reconstruction from $M$ coefficients including the slide's $M=4$, and the exact identity 'mean reconstruction error = sum of discarded eigenvalues'. Note: $361\times261 = 94{,}221$, not 83,781 (which equals $321\times261$), so one of the two numbers on slide 55 contains a typo; the idea is unaffected.
- **Exam tip.** Reconstruction is $\hat x = \bar x + \sum_{k\le M} q_ku_k$ — do not forget to add the mean back. Expect to compute the storage saving ($K$ numbers → $M$ coefficients, plus the shared $\bar x$ and $U_M$).

---
## 10 · Limits of PCA — and how to read a PCA plot honestly

> 📍 **Where we are.** PCA worked beautifully on the toy, the simulated map and the digits. Two quick experiments show
> where it can mislead you.

PCA is **linear** (each PC is a weighted sum of features) and it ranks directions by **variance**, which is not the same
as biological importance. It is also **non-probabilistic** — in the language of the course's two-step template it has
no explicit model of the data; its probabilistic cousin, *probabilistic PCA* (Tipping & Bishop 1999), assumes a latent
Gaussian variable $z$ with $x = Wz + \mu + \varepsilon$ and can be fitted by EM — the latent-variable idea of Module 13.

In [ ]:
# ── 10.1 PCA is linear: cells going round the cell cycle (SIMULATED) ──
rng = np.random.default_rng(SEED + 10)
phase = rng.uniform(0, 2 * np.pi, 300)                    # position of each cell in the cycle (0 … 2π)
peaks = np.array([0.0, 2 * np.pi / 3, 4 * np.pi / 3])     # three marker genes peaking at different phases
X_cyc = np.cos(phase[:, None] - peaks[None, :]) + rng.normal(0, 0.15, (300, 3))
_, U_cyc, lam_cyc, sc_cyc = pca_eig(X_cyc, 2)
rho = stats.spearmanr(sc_cyc[:, 0], phase)[0]
angle = np.arctan2(sc_cyc[:, 1], sc_cyc[:, 0])            # a NON-linear function of the two PC scores
print("% variance PC1–PC3:", np.round(100 * lam_cyc / lam_cyc.sum(), 1))
print(f"rank correlation of the PC1 score with the cell-cycle phase: ρ = {rho:+.2f}")

fig, axes = plt.subplots(1, 3, figsize=(12, 3.3))
axes[0].scatter(sc_cyc[:, 0], sc_cyc[:, 1], c=phase, cmap="twilight", s=8)
axes[0].set_aspect("equal"); axes[0].set_xlabel("PC1 score"); axes[0].set_ylabel("PC2 score")
axes[0].set_title("PCA finds the plane of the loop")
axes[1].scatter(phase, sc_cyc[:, 0], c=phase, cmap="twilight", s=8)
axes[1].set_xlabel("true cell-cycle phase (radians)"); axes[1].set_ylabel("PC1 score")
axes[1].set_title("…but no single PC orders the cells")
axes[2].scatter(phase, angle, c=phase, cmap="twilight", s=8)
axes[2].set_xlabel("true cell-cycle phase (radians)"); axes[2].set_ylabel("angle in PC1–PC2 plane (wraps at ±π)")
axes[2].set_title("the angle (non-linear) does")
plt.tight_layout(); plt.show()

In [ ]:
# ── 10.2 Variance ≠ biological relevance: a batch effect dominates PC1 (SIMULATED) ──
rng = np.random.default_rng(SEED + 11)
N_b, K_b = 120, 300
batch = np.repeat([0, 1], N_b // 2)                       # two processing days
subtype = rng.permutation(np.repeat([0, 1], N_b // 2))    # two tumour subtypes, balanced across days
X_b = rng.normal(0, 1, (N_b, K_b))
X_b[:, :100] += 1.2 * batch[:, None]                      # technical shift in 100 genes
X_b[:, 100:115] += 1.2 * subtype[:, None]                 # biological shift in 15 genes
lam_b, sc_b, _ = svd_pca(X_b)
for k in range(3):
    print(f"PC{k + 1} ({100 * lam_b[k] / lam_b.sum():.1f} %): batch separation {abs(class_separation(sc_b[:, k], batch)):.2f} SD, "
          f"subtype separation {abs(class_separation(sc_b[:, k], subtype)):.2f} SD")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
for ax, lab, name in [(axes[0], batch, "processing day"), (axes[1], subtype, "tumour subtype")]:
    for c in (0, 1):
        ax.scatter(sc_b[lab == c, 0], sc_b[lab == c, 1], s=12, color=CLASS_COLORS[c], label=f"{name} {c}")
    ax.set_xlabel("PC1 score"); ax.set_ylabel("PC2 score"); ax.legend(fontsize=7); ax.set_title(f"coloured by {name}")
plt.tight_layout(); plt.show()

🔍 **What just happened?** (1) Cells progressing through the cell cycle trace a **loop**: three marker genes peak one
after another. PCA does its job — PC1 and PC2 hold almost all the variance and show the ring — but no single linear
coordinate orders the cells: the PC1 score is a wave over the cycle, so cells at quite different phases share the same
score, and its rank correlation with the phase is only moderate (|ρ| ≈ 0.45 in the printout). The biologically meaningful coordinate, the phase,
is the *angle* in the PC plane — a non-linear function of the data. Curved and branching structures (cell cycles,
differentiation trajectories) need non-linear methods (diffusion maps, t-SNE/UMAP for visualisation, autoencoders —
[Module 22](22_neural_networks.ipynb)). (2) A technical batch effect that touches many genes produces the largest variance, so PC1 separates
processing days, while the real biological signal, carried by only 15 genes, is pushed to PC2 (printout). Had we looked
only at PC1, we would have "discovered" the batches.

⚠️ **Reading a PCA plot honestly.**
- PC1 is the largest source of **variance**, not necessarily the most interesting biology — check for batches, sample
  quality and global effects (Section 8.2 showed a real one).
- **Scaling is a choice** (centred vs standardised, Section 8): report it.
- Signs and rotations of PCs are arbitrary; distances between groups in a 2-D plot ignore the other $K-2$ directions.
- Separation seen in a PCA plot is **hypothesis-generating**: confirm it with an independent test, new samples or an
  experiment.

---
## 📌 Key takeaways

- **PCA is unsupervised dimensionality reduction**: it re-describes $N$ samples with $M \ll K$ new features that preserve
  as much of the data's variability as possible (slide 59).
- **Step 1** — centre $X$, form $S = \frac1N X^\top X$, eigendecompose $S = UDU^\top$ (`np.linalg.eigh`, then sort
  descending). Eigenvectors $u_k$ are the principal directions; eigenvalues $\lambda_k$ are the variances along them.
- **Step 2** — project: scores $X_p = X[u_1,\dots,u_M]$; reconstruct with $\hat x = \bar x + \sum_k q_ku_k$. The average
  reconstruction error is the sum of the discarded eigenvalues. Always centre with the training mean.
- **Choose $M$** with a scree plot or a target fraction $\sum_{k\le M}\lambda_k/\sum_k\lambda_k$; compare with a null
  (shuffled) scree plot to see which PCs stand out of the noise.
- **Why it works** — maximising $u^\top Su$ subject to $u^\top u=1$ gives, via the Lagrangian, $Su=\lambda u$; the variance
  equals $\lambda$, so the top eigenvector wins. PCs are orthogonal and their scores uncorrelated.
- **Use the SVD** $X=V\Sigma W^\top$ when $K\gg N$: $W$ = directions, $\lambda_k=\sigma_k^2/N$; at most $N-1$ non-zero PCs.
- **Biology** — genotype PCs can mirror geography (isolation by distance); expression PCs can show disease classes *or*
  technical effects depending on scaling; images decompose into eigen-images.
- **Limits** — linear, variance-driven, scaling-dependent and non-probabilistic: PCA plots generate hypotheses, they do
  not prove them.

---
## ✅ Checkpoint

1. `np.linalg.eigh(S)` returns `evals = [0.2, 1.1, 4.7]` and a matrix `evecs`. Which column is PC1, and what fraction of the variance does it explain?
   <details><summary>Answer</summary>The last column, `evecs[:, 2]`, because `eigh` sorts eigenvalues in ascending order. Fraction $4.7/(0.2+1.1+4.7) = 4.7/6.0 \approx 0.78$.</details>
2. For centred data with covariance $S=\begin{pmatrix}3&1\\1&3\end{pmatrix}$, find the eigenvalues and the first PC by hand.
   <details><summary>Answer</summary>Characteristic equation $(3-\lambda)^2-1=0 \Rightarrow \lambda=4$ or $2$. For $\lambda=4$: $(S-4I)u=0$ gives $u_1=u_2$, so $u_1=(1,1)/\sqrt2$. PC1 explains $4/6=2/3$ of the variance; PC2 is $(1,-1)/\sqrt2$.</details>
3. Why must the data be mean-centred before PCA (slide 36's 'why?')?
   <details><summary>Answer</summary>Because $\frac1N X^\top X$ is the covariance only for centred data. Without centring it mixes the location of the cloud with its spread, and the top eigenvector points towards the mean instead of along the direction of largest variation (Section 4.3).</details>
4. Write down the Lagrangian for PCA, differentiate it, and explain why the solution is the eigenvector with the **largest** eigenvalue.
   <details><summary>Answer</summary>$\mathcal L = u^\top Su + \lambda(1-u^\top u)$; $\partial\mathcal L/\partial u = 2Su-2\lambda u = 0 \Rightarrow Su=\lambda u$, so $u$ is an eigenvector. Its variance is $u^\top Su = \lambda u^\top u = \lambda$, which is maximised by the largest eigenvalue.</details>
5. A gene-expression matrix has $N=50$ samples and $K=20{,}000$ genes. How many non-zero eigenvalues can the covariance matrix have, and how would you compute the PCs efficiently?
   <details><summary>Answer</summary>At most $N-1=49$ (centred data span at most a 49-dimensional space). Use the SVD of the centred $50\times20{,}000$ matrix (or eigendecompose the $50\times50$ matrix $XX^\top$) instead of the $20{,}000\times20{,}000$ covariance.</details>
6. Eigenvalues of a 64-pixel image data set sum to 1,200, and the first 10 sum to 900. What is the average squared reconstruction error with $M=10$?
   <details><summary>Answer</summary>The sum of the discarded eigenvalues: $1{,}200-900 = 300$ (per image, in squared grey levels).</details>
7. In the Golub data, PC1 of the *standardised* genes has a correlation of about 0.96 (in absolute value — the sign of a PC is arbitrary) with each array's mean expression. What does this tell you, and what should you not conclude?
   <details><summary>Answer</summary>The largest variance direction after standardising is a global, probably technical, sample-level effect (array brightness), not leukaemia type. Do not conclude that ALL/AML is unimportant, or that PC1 is biology; the leading PCs reflect whatever dominates the variance under the chosen scaling.</details>
8. What does this code compute? `q = (x - X.mean(0)) @ U[:, :4]; x_hat = X.mean(0) + U[:, :4] @ q`
   <details><summary>Answer</summary>The four PC coefficients $[q_1,\dots,q_4]$ of the sample $x$ (slide 58) and its reconstruction from those four coefficients — the best approximation of $x$ in the 4-dimensional PC subspace.</details>

## 🧠 Questions & discussion

1. Novembre et al. found that two PCs of 197,146 SNPs resemble the map of Europe. What sampling choices (who was genotyped, from where, how many per country) could change or distort that picture?
2. A colleague shows you a PCA plot in which treated and control cell cultures separate perfectly along PC1. List three questions you would ask before believing that the treatment changed gene expression.
3. When would you standardise genes before PCA, and when would you only centre them? Use the Golub results of Section 8 in your answer.
4. PCA chooses directions of maximal variance without looking at labels. Give an example in which the direction that best separates two classes has *small* variance, so that PCA followed by a classifier would fail.
5. PCA is the non-probabilistic member of the course's method map. What would you gain from probabilistic PCA (a likelihood, missing-data handling, model selection by BIC — Module 23)?

## 🚀 Bonus — going further

- **Power iteration and deflation.** Implement $u\leftarrow Su/\lVert Su\rVert$, then remove the found component ($S\leftarrow S-\lambda_1u_1u_1^\top$) and repeat to obtain $u_2, u_3$; compare with `eigh`.
- **PCs as GWAS covariates.** Using Module 06's regression, simulate a phenotype that differs between the east and west of the Section-7 map but has *no* causal SNP; scan all SNPs with and without the top two PCs as covariates and compare the number of false 'hits' (the idea of Price et al. 2006).
- **Linear autoencoder = PCA.** In Module 22 a network with a linear bottleneck of width $M$ learns the same subspace as the top $M$ PCs; train one on the digits and compare reconstruction errors with Section 9.
- **PCA before clustering.** Repeat the Golub clustering of Module 12 on the top 2, 5 and 20 PC scores instead of all genes; how does the agreement with ALL/AML (adjusted Rand index) change?
- **Sister course.** The cryo-ET curriculum [`BiologyToResearch`](../../../../cryoem/tutorial/BiologyToResearch/) uses PCA followed by a Gaussian mixture to sort subtomograms of macromolecules without labels — the same two tools as Modules 13–14, applied to 3-D images.

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module: 14 — Principal component analysis
Input: N × K data matrix (2-D toy, 10-gene toy, simulated genotypes 480 × ~3,000, Golub 72 × 3,571, digits 1,797 × 64)
Output: eigenvalues (variance per PC), principal directions u_k, PC scores, reconstructions
Method: centre → S = XᵀX/N → eigh (sort descending) or SVD → project onto top M directions
Key equation: S u = λ u  (from max uᵀSu s.t. uᵀu = 1);  X_p = X [u1 … uM];  error = Σ_{k>M} λ_k
Key parameters: M (scree plot / % variance); centring and scaling choice; SNP standardisation
What changed when I modified … (noise level in 5.1, m_rate or grid shape in 7.1, scaling in 8.2):
One failure mode: (e.g. PC1 tracks array brightness or a batch, not biology; a cell-cycle loop has no linear order)
Connection to 02-620 slides: 011_PCA slides 17–31 (Step 1), 32–40 (Step 2), 41–44 (choosing M), 47–54 (derivation)
```

## 📚 References

- Carnegie Mellon University 02-620 *Machine Learning for Scientists*, Spring 2025 lecture slides, deck `011_PCA_mz_031825` (course design S. Kim; instructors M. Xu and M. Zhang).
- Murphy K. P. (2012) *Machine Learning: A Probabilistic Perspective*. MIT Press — Chapter 12 (latent linear models; §12.2 PCA, probabilistic PCA).
- Hastie T., Tibshirani R., Friedman J. (2009) *The Elements of Statistical Learning*, 2nd ed. Springer — §14.5.1 (principal components) and §3.5.1 (principal components regression).
- Pearson K. (1901) On lines and planes of closest fit to systems of points in space. *Philosophical Magazine* 2:559–572. · Hotelling H. (1933) Analysis of a complex of statistical variables into principal components. *Journal of Educational Psychology* 24:417–441.
- Novembre J., Johnson T., Bryc K., et al. (2008) Genes mirror geography within Europe. *Nature* 456:98–101.
- Novembre J., Stephens M. (2008) Interpreting principal component analyses of spatial population genetic variation. *Nature Genetics* 40:646–649.
- Patterson N., Price A. L., Reich D. (2006) Population structure and eigenanalysis. *PLoS Genetics* 2:e190.
- Price A. L., Patterson N. J., Plenge R. M., et al. (2006) Principal components analysis corrects for stratification in genome-wide association studies. *Nature Genetics* 38:904–909.
- Kimura M., Weiss G. H. (1964) The stepping stone model of population structure and the decrease of genetic correlation with distance. *Genetics* 49:561–576.
- Golub T. R., Slonim D. K., Tamayo P., et al. (1999) Molecular classification of cancer: class discovery and class prediction by gene expression monitoring. *Science* 286:531–537. · Dudoit S., Fridlyand J., Speed T. P. (2002) Comparison of discrimination methods for the classification of tumors using gene expression data. *JASA* 97:77–87.
- Tipping M. E., Bishop C. M. (1999) Probabilistic principal component analysis. *Journal of the Royal Statistical Society B* 61:611–622.
- Turk M., Pentland A. (1991) Eigenfaces for recognition. *Journal of Cognitive Neuroscience* 3:71–86. · Digits: Alpaydin E., Kaynak C. (1998) Optical Recognition of Handwritten Digits, UCI Machine Learning Repository (as shipped with scikit-learn).

## ➡️ Next

Part C is complete: you can cluster (Module 12), fit mixtures with EM (Module 13) and reduce dimension (this module).
Part D turns to **probabilistic graphical models**: [Module 15 — Bayesian networks I](15_bayesian_networks_representation.ipynb)
shows how a graph of conditional independences turns an impossible joint distribution into a handful of small tables.